In [1]:
# ================= ONE-CELL: robust COCO->YOLO-seg + TRAIN ON T4 =================
!nvidia-smi
!pip -q install "ultralytics>=8.3.0" pycocotools opencv-python-headless

from ultralytics import YOLO
import os, json, glob, shutil, random, pathlib, yaml, zipfile
import numpy as np, cv2
from pycocotools import mask as coco_mask
from google.colab import drive

drive.mount('/content/drive')

# ---------- CONFIG (edit IMG_ROOT if needed) ----------
ROOT      = "/content/drive/MyDrive/colon"                     # where your annotations live
ANN_DIR   = os.path.join(ROOT, "coco_annotation")              # contains instances_default.json
IMG_ROOT  = "/content/drive/MyDrive/colon/kvasir-dataset-v2"   # nested folders (polyps, normal-*, etc.)
ALT_IMG   = "/content/drive/MyDrive/kvasir-dataset-v2"
if not os.path.exists(IMG_ROOT) and os.path.exists(ALT_IMG):
    IMG_ROOT = ALT_IMG
# ------------------------------------------------------

# 1) normalize annotation folder name (avoid spaces)
RAW_ANN_DIR_WITH_SPACES = os.path.join(ROOT, "coco annotation")
if os.path.exists(RAW_ANN_DIR_WITH_SPACES) and not os.path.exists(ANN_DIR):
    os.rename(RAW_ANN_DIR_WITH_SPACES, ANN_DIR)

COCO_JSON = os.path.join(ANN_DIR, "instances_default.json")
assert os.path.exists(COCO_JSON), f"Missing COCO json: {COCO_JSON}"
assert os.path.exists(IMG_ROOT),  f"Image root not found: {IMG_ROOT}"

# 2) unzip any archives once (no-op if already extracted)
for z in glob.glob(os.path.join(ROOT, "*.zip")):
    out_dir = os.path.join(ROOT, pathlib.Path(z).stem)
    if not os.path.exists(out_dir):
        print(f"Unzipping {z} -> {out_dir}")
        with zipfile.ZipFile(z, "r") as zf:
            zf.extractall(out_dir)

# 3) clean COCO (strip dirs from file_name so basenames match disk)
CLEAN_DIR  = "/content/coco_ann_clean"
shutil.rmtree(CLEAN_DIR, ignore_errors=True)
os.makedirs(CLEAN_DIR, exist_ok=True)
CLEAN_JSON = os.path.join(CLEAN_DIR, "instances_default.json")

with open(COCO_JSON, "r", encoding="utf-8") as f:
    coco = json.load(f)

for im in coco.get("images", []):
    im["file_name"] = os.path.basename(im["file_name"])

with open(CLEAN_JSON, "w", encoding="utf-8") as f:
    json.dump(coco, f)
print("Cleaned COCO ->", CLEAN_JSON)

# 4) index all images (nested) so we can read sizes & copy later
IMG_EXTS = {".jpg",".jpeg",".png",".bmp",".tif",".tiff"}
all_imgs = [p for p in glob.glob(os.path.join(IMG_ROOT, "**", "*"), recursive=True)
            if pathlib.Path(p).suffix.lower() in IMG_EXTS]
print(f"Discovered {len(all_imgs)} images under {IMG_ROOT}")

fname_to_img = {pathlib.Path(p).name.lower(): p for p in all_imgs}
stem_to_img  = {pathlib.Path(p).stem: p for p in all_imgs}

# 5) build YOLO-seg labels DIRECTLY from COCO via frPyObjects -> decode -> contours
images = {im["id"]: im for im in coco["images"]}
cats   = sorted(coco.get("categories", []), key=lambda c: c.get("id", 0))
cat_id_to_idx = {c["id"]: i for i, c in enumerate(cats)}  # 0..nc-1

anns_by_img = {}
for a in coco.get("annotations", []):
    anns_by_img.setdefault(a["image_id"], []).append(a)

def resolve_image_path(im):
    fn = os.path.basename(im["file_name"])
    return fname_to_img.get(fn.lower()) or stem_to_img.get(pathlib.Path(fn).stem)

def get_image_size(im):
    w, h = im.get("width"), im.get("height")
    if w and h:
        return int(w), int(h)
    p = resolve_image_path(im)
    if p and os.path.exists(p):
        arr = cv2.imread(p, cv2.IMREAD_UNCHANGED)
        if arr is not None:
            hh, ww = arr.shape[:2]
            return int(ww), int(hh)
    raise RuntimeError(f"Could not determine size for image {im.get('file_name')}")

def is_polygon_seg(seg):
    return isinstance(seg, list) and (len(seg) == 0 or isinstance(seg[0], list))

def is_rle_list(seg):
    return isinstance(seg, list) and len(seg) > 0 and isinstance(seg[0], dict) and "counts" in seg[0]

def is_rle_dict(seg):
    return isinstance(seg, dict) and "counts" in seg

def polygons_normalized_to_pixel(seg_list, w, h):
    """If polygon coords look normalized (<=1), scale to pixels."""
    fixed = []
    for poly in seg_list:
        arr = np.asarray(poly, dtype=np.float32)
        if arr.size < 6:
            continue
        xs, ys = arr[0::2].copy(), arr[1::2].copy()
        if (np.nanmax(xs) <= 1.001 and np.nanmax(ys) <= 1.001):
            xs *= (w - 1); ys *= (h - 1)
        fixed.append(np.column_stack([xs, ys]).reshape(-1).tolist())
    return fixed

def seg_to_mask(seg, h, w):
    """
    Robustly convert any COCO 'segmentation' (polygons, RLE dict, list-of-RLE) to a binary mask.
    Always goes through frPyObjects first to normalize uncompressed RLE (list counts) etc.
    """
    try:
        if is_polygon_seg(seg):
            seg_fixed = polygons_normalized_to_pixel(seg, w, h)
            if not seg_fixed:
                return None
            rle = coco_mask.frPyObjects(seg_fixed, h, w)

        elif is_rle_dict(seg) or is_rle_list(seg):
            # frPyObjects normalizes both compressed ('counts' string) and uncompressed (list) RLEs
            rle = coco_mask.frPyObjects(seg, h, w)

        else:
            return None

        m = coco_mask.decode(rle)
        # unify to HxW
        if m.ndim == 3:
            m = np.any(m, axis=2).astype(np.uint8)
        else:
            m = (m > 0).astype(np.uint8)

        # fix shape if needed
        if m.shape != (h, w):
            if m.shape == (w, h):  # swapped
                m = m.T
            else:
                m = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
        return m
    except Exception as e:
        # Any decode hiccup -> skip this instance
        return None

def mask_to_yolo_polylines(m, w, h, approx_eps=2.0, min_area=5):
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    lines = []
    for c in cnts:
        if cv2.contourArea(c) < min_area:
            continue
        c = cv2.approxPolyDP(c, approx_eps, True)
        if c.shape[0] < 3:
            continue
        c = c.squeeze(1).astype(np.float32)
        xs = np.clip(c[:, 0], 0, max(1, w-1)) / max(1, w)
        ys = np.clip(c[:, 1], 0, max(1, h-1)) / max(1, h)
        coords = np.column_stack([xs, ys]).reshape(-1)
        if len(coords) >= 6:
            lines.append(' '.join(f'{v:.6f}' for v in coords))
    return lines

LBL_ROOT = "/content/yolo_from_coco_labels"
shutil.rmtree(LBL_ROOT, ignore_errors=True)
os.makedirs(LBL_ROOT, exist_ok=True)

non_empty, empty, inst_total = 0, 0, 0
for img_id, im in images.items():
    fn = os.path.splitext(os.path.basename(im["file_name"]))[0]
    try:
        w, h = get_image_size(im)
    except Exception:
        empty += 1
        continue

    lines_out = []
    for a in anns_by_img.get(img_id, []):
        seg = a.get("segmentation")
        if seg is None:
            continue
        m = seg_to_mask(seg, h, w)
        if m is None or m.max() == 0:
            continue
        polys = mask_to_yolo_polylines(m, w, h, approx_eps=2.0, min_area=5)
        if not polys:
            continue
        cid = cat_id_to_idx[a["category_id"]]
        for poly in polys:
            lines_out.append(f"{cid} {poly}")
            inst_total += 1

    out = os.path.join(LBL_ROOT, f"{fn}.txt")
    with open(out, "w") as f:
        for ln in lines_out:
            f.write(ln + "\n")
    if lines_out: non_empty += 1
    else:         empty += 1

print(f"Labels written -> non-empty files: {non_empty}, empty files: {empty}, instances total: {inst_total}  (dir: {LBL_ROOT})")

# 6) pair ONLY non-empty labels to images; build /content/yolo_ds
DS = "/content/yolo_ds"
for p in ["images/train","images/val","labels/train","labels/val"]:
    os.makedirs(os.path.join(DS, p), exist_ok=True)

pairs = []
for lbl in glob.glob(os.path.join(LBL_ROOT, "*.txt")):
    if os.path.getsize(lbl) == 0:
        continue
    stem = pathlib.Path(lbl).stem
    found = None
    for ext in (".jpg",".jpeg",".png",".bmp",".tif",".tiff"):
        cand = fname_to_img.get((stem+ext).lower())
        if cand: found = cand; break
    if not found:
        found = stem_to_img.get(stem)
    if found:
        pairs.append((found, lbl))

print(f"Positive pairs (used for training): {len(pairs)}")
assert pairs, "No positive pairs found. Open a file in /content/yolo_from_coco_labels/ and confirm it now has polygon lines."

random.seed(0); random.shuffle(pairs)
cut = max(1, int(0.85 * len(pairs)))
train_pairs, val_pairs = pairs[:cut], pairs[cut:]

def cp(pairs, split):
    for img,lbl in pairs:
        shutil.copy2(img, os.path.join(DS, "images", split, pathlib.Path(img).name))
        shutil.copy2(lbl, os.path.join(DS, "labels", split, pathlib.Path(lbl).name))

cp(train_pairs, "train"); cp(val_pairs, "val")
print(f"Dataset -> train: {len(train_pairs)}, val: {len(val_pairs)}")
!find /content/yolo_ds -maxdepth 2 -type d -print

# 7) dataset.yaml from categories
names = [c.get("name", f"class_{i}") for i,c in enumerate(cats)] or ["object"]
with open(os.path.join(DS, "dataset.yaml"), "w") as f:
    yaml.safe_dump({"path": DS, "train": "images/train", "val": "images/val", "names": names}, f, sort_keys=False)
print("Classes:", names)

# 8) Train on T4 (segmentation)
!yolo task=segment mode=train \
     model=yolo11s-seg.pt data=/content/yolo_ds/dataset.yaml \
     imgsz=640 epochs=60 batch=-1 device=0 workers=2 cache=True \
     project=runs/segment name=train_t4 exist_ok=True

# 9) Inference
import glob as _g
best = sorted(_g.glob("/content/runs/segment/train_t4*/weights/best.pt"))[-1]
print("Using weights:", best)
YOLO(best).predict(source=os.path.join(DS, "images", "val"), save=True, device=0, imgsz=640)
# =====================================================================


Wed Aug 27 06:54:22 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   70C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'lumen'}
 obb: None
 orig_img: array([[[  0,   0,   0],
         [  0,   0,   0],
         [  0,   0,   0],
         ...,
         [  0,   0,   0],
         [  0,   0,   0],
         [  0,   0,   0]],
 
        [[  0,   0,   0],
         [  0,   0,   0],
         [  0,   0,   0],
         ...,
         [  0,   0,   0],
         [  0,   0,   0],
         [  0,   0,   0]],
 
        [[  0,   0,   0],
         [  0,   0,   0],
         [  0,   0,   0],
         ...,
         [  0,   0,   0],
         [  0,   0,   0],
         [  0,   0,   0]],
 
        ...,
 
        [[  0,   0,   1],
         [  4,   0,   1],
         [  3,   1,   1],
         ...,
         [  0,   0,   1],
         [  0,   0,   0],
         [  2,   0,   0]],
 
        [[ 22, 227, 254],
         [  0,   2, 181],
         [ 19, 225, 253],
         ...,
         [  1,   2,

# NOTE** the above only trains on labled images but no negatives. once i finish going through the data set then edit so it looks at negatives too*** **bold text**

In [2]:
from ultralytics import YOLO, settings
import glob

best = sorted(glob.glob("/content/runs/segment/*/weights/best.pt"))[-1]
print("Exporting:", best)

# ONNX for broad compatibility
!yolo export model="{best}" format=onnx opset=12 dynamic=True device=0

# (Optional) TensorRT for fastest NVIDIA inference
# !yolo export model="{best}" format=engine device=0 half=True


Exporting: /content/runs/segment/train_t4/weights/best.pt
Ultralytics 8.3.186 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11s-seg summary (fused): 113 layers, 10,067,203 parameters, 0 gradients, 32.8 GFLOPs

PyTorch: starting from '/content/runs/segment/train_t4/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) ((1, 37, 8400), (1, 32, 160, 160)) (19.6 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<1.18.0', 'onnxslim>=0.1.59', 'onnxruntime-gpu'] not found, attempting AutoUpdate...

requirements: AutoUpdate success ✅ 7.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.17.0 opset 12...
ONNX: slimming with onnxslim 0.1.65...
ONNX: export success ✅ 13.7s, saved as '/content/runs/segment/train_t4/weights/best.onnx' (38.

In [5]:
# --- Upload a picture and run inference with your latest best.pt ---
from google.colab import files
from ultralytics import YOLO
import glob, os
from IPython.display import Image, display

# 1) Upload from your computer
UPLOAD_DIR = "/content/test_uploads"
os.makedirs(UPLOAD_DIR, exist_ok=True)
uploaded = files.upload()  # choose 1+ images (jpg/png/jpeg)
for name, _ in uploaded.items():
    os.rename(name, os.path.join(UPLOAD_DIR, os.path.basename(name)))

# 2) Load the freshest trained weights
best = sorted(glob.glob("/content/runs/segment/*/weights/best.pt"))[-1]
print("Using weights:", best)
model = YOLO(best)

# 3) Predict on uploaded images (saves annotated outputs)
OUT_DIR = "/content/preds_uploads"
results = model.predict(source=UPLOAD_DIR, save=True, device=0, imgsz=640, project=OUT_DIR, name="")
print("Saved predictions to:", OUT_DIR)

# 4) Preview a few results
pred_paths = sorted(glob.glob(os.path.join(OUT_DIR, "*.jpg"))) or sorted(glob.glob(os.path.join(OUT_DIR, "*.*")))
for p in pred_paths[:8]:
    print(os.path.basename(p))
    display(Image(filename=p))


Saving TerminalIlumRetroflexed.mpg to TerminalIlumRetroflexed.mpg
Saving Colonoscopy2.mpg to Colonoscopy2.mpg
Saving Colonoscopy.mpg to Colonoscopy.mpg
Using weights: /content/runs/segment/train_t4/weights/best.pt

WARNING ⚠️ 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/6 /content/test_uploads/images-1.jpeg: 480x640 1 lumen, 17.7ms
image 2/6 /content/test_uploads/images-2.jpeg: 576x640 2 lumens, 20.0ms
video 3/6 (frame 1/606) /content/test_uploads/Colonoscopy.mpg: 448x640 (no detections), 68.8ms
vide